# Model 1: Simple CNN

## 1. Cấu hình và thư viện

In [ ]:
import os
import hashlib
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, metrics, callbacks
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
tf.keras.utils.set_random_seed(SEED) 


DATA_DIR = Path("/home/pc/AAA/project/deepnn/G8_AI66A_DNN/data/raw")
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR = DATA_DIR / "test"

SETS = ["train", "val", "test"]
IMG_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 30
NUM_CLASSES = 4
AUTOTUNE = tf.data.AUTOTUNE

# Thứ tự lớp theo alphabet (giống image_dataset_from_directory)
class_names = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
assert len(class_names) == NUM_CLASSES, class_names
print(f"Các lớp: {class_names}")

## 2. Preprocessing

### 2.1 Loại bỏ ảnh trùng lặp

In [ ]:
clean_samples = {se: [] for se in SETS}
seen_hash = {}               # hash -> (split, class) của bản được giữ lại
dup_count = Counter()        # (split bị loại, split giữ lại) -> số ảnh
label_conflicts = 0          # cùng nội dung nhưng khác nhãn

# Duyệt test -> val -> train: ảnh trùng luôn bị loại khỏi tập có ưu tiên thấp hơn,
# nên test được giữ sạch nhất. sorted() để kết quả ổn định giữa các lần chạy.
for se in ["test", "val", "train"]:
    se_dir = DATA_DIR / se

    for class_name in sorted(os.listdir(se_dir)):
        class_dir = se_dir / class_name
        if not class_dir.is_dir():
            continue

        for img_path in sorted(class_dir.iterdir()):
            if not img_path.is_file():
                continue

            # Tính mã MD5 của file ảnh
            file_hash = hashlib.md5(img_path.read_bytes()).hexdigest()

            if file_hash in seen_hash:
                kept_in, kept_cls = seen_hash[file_hash]
                dup_count[(se, kept_in)] += 1
                if kept_cls != class_name:
                    label_conflicts += 1
            else:
                seen_hash[file_hash] = (se, class_name)
                clean_samples[se].append((img_path, class_name))

print("Ảnh trùng bị loại:", sum(dup_count.values()))
for (drop_in, kept_in), n in dup_count.items():
    print(f"- {n} ảnh ở '{drop_in}' trùng với ảnh ở '{kept_in}'")
print("Ảnh trùng nội dung nhưng khác nhãn:", label_conflicts)

del seen_hash  # Giải phóng bộ nhớ đệm hash sau khi dùng xong

clean_summary = pd.DataFrame({s: Counter(class_name for _, class_name in clean_samples[s]) for s in SETS}).fillna(0).astype(int)
clean_summary["total"] = clean_summary.sum(axis=1)
clean_summary.loc["TOTAL"] = clean_summary.sum()
print("\nSau khi làm sạch:")
clean_summary

### 2.2 Tạo dataset từ dữ liệu đã lọc trùng

Dataset được dựng trực tiếp từ `clean_samples` (thay cho `image_dataset_from_directory` trên thư mục gốc), để train/val/test thật sự không còn ảnh trùng nhau.

In [ ]:
class_to_idx = {c: i for i, c in enumerate(class_names)}


def load_image(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))  # float32, thang [0, 255]
    return img, tf.one_hot(label, NUM_CLASSES)


def make_ds(samples, shuffle=False):
    paths = [str(p) for p, _ in samples]
    labels = [class_to_idx[c] for _, c in samples]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_ds(clean_samples["train"], shuffle=True)
val_ds = make_ds(clean_samples["val"])
test_ds = make_ds(clean_samples["test"])

for name in SETS:
    print(f"{name}: {len(clean_samples[name])} ảnh")

### 2.3 Kiểm tra mất cân bằng lớp

In [ ]:
# Tạo DataFrame từ kết quả đếm để vẽ biểu đồ cho đẹp
data_summary = []
for split in SETS:
    # Đếm số lượng ảnh của từng lớp trong biến clean_samples
    split_counts = pd.Series([cls for _, cls in clean_samples[split]]).value_counts()
    for class_name in class_names:
        count = split_counts.get(class_name, 0)
        data_summary.append({"Split": split, "Class": class_name, "Count": count})
        
df_summary = pd.DataFrame(data_summary)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_summary, x="Class", y="Count", hue="Split", palette="Set2")
plt.title("Phân bố số lượng ảnh các lớp (Đã lọc trùng)", fontsize=14, fontweight="bold")
plt.ylabel("Số lượng ảnh", fontsize=12)
plt.xlabel("Lớp (Class)", fontsize=12)
plt.legend(title="Tập dữ liệu")
plt.tight_layout()
plt.show()

### 2.4 Tính Mean và Std cho normalization

In [ ]:
# Tính Mean và Std theo từng kênh RGB trên tập train
# Ảnh trong train_ds có dạng (batch, height, width, channels)
channel_sum = np.zeros(3, dtype=np.float64)
channel_squared_sum = np.zeros(3, dtype=np.float64)
pixel_count = 0

for images, _ in train_ds:
    # Chuyển pixel từ [0, 255] về [0, 1]
    images = tf.cast(images, tf.float32) / 255.0

    # Cộng giá trị pixel theo từng kênh RGB
    channel_sum += tf.reduce_sum(
        images, axis=[0, 1, 2]
    ).numpy()

    # Cộng bình phương giá trị pixel theo từng kênh RGB
    channel_squared_sum += tf.reduce_sum(
        tf.square(images), axis=[0, 1, 2]
    ).numpy()

    # Đếm tổng số pixel đã xử lý trong mỗi kênh
    pixel_count += int(tf.shape(images)[0]) * IMG_SIZE * IMG_SIZE

# Mean của từng kênh
MEAN = channel_sum / pixel_count
# Phương sai = E[X^2] - E[X]^2
variance = channel_squared_sum / pixel_count - np.square(MEAN)
# Std = căn bậc hai của phương sai
STD = np.sqrt(np.maximum(variance, 0.0))

print("Mean (R, G, B):", MEAN.tolist())
print("Std  (R, G, B):", STD.tolist())

## 3. Xây dựng model

In [ ]:
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(15 / 360),
    layers.RandomZoom(
        height_factor=(-0.1, 0.1),
        width_factor=(-0.1, 0.1)
    ),
    layers.RandomBrightness(factor=0.2),
    layers.RandomContrast(factor=0.2),
    layers.RandomSaturation(factor=0.2),

    # Chuyển pixel từ [0, 255] sang [0, 1]
    layers.Rescaling(1.0 / 255),

    # MEAN và STD đã được tính trên train_ds ở thang [0, 1]
    layers.Normalization(
        axis=-1,
        mean=MEAN.astype("float32"),
        variance=np.square(STD.astype("float32"))
    ),

    
    # CNN    
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])


In [ ]:

model.summary()

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', metrics.F1Score(average='macro', name='f1_score')]
)

cb_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

## 4. Huấn luyện

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=cb_list
)

## 5. Đánh giá trên tập test

In [ ]:
y_true = np.concatenate([y for x, y in test_ds], axis=0)
y_true = np.argmax(y_true, axis=1)

y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

In [ ]:
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Vẽ Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Lưu model
model.save('model1_simple_cnn.keras')
print("Đã lưu model thành công!")